In [1]:
import os
import time
import math
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from tqdm import tqdm

import igraph as ig
import leidenalg as la
from sklearn.metrics import adjusted_rand_score
from itertools import combinations
import pickle

#### _def functions

In [ ]:
def build_graph_and_clustering(
    edges_df,
    gene_col1="gene1",
    gene_col2="gene2",
    weight_col="dif",
    resolution=0.05,
    seed=69,
):
    relevant_df = edges_df[[gene_col1, gene_col2, weight_col]].copy()
    relevant_df[weight_col] = relevant_df[weight_col].astype(float)

    G = ig.Graph.DataFrame(relevant_df, directed=False, use_vids=False)
    G.simplify(combine_edges="last")

    if weight_col != "weight":
        G.es["weight"] = G.es[weight_col]
        del G.es[weight_col]

    partition = la.find_partition(
        G,
        la.RBConfigurationVertexPartition,
        weights="weight",
        resolution_parameter=resolution,
        seed=seed,
    )
    clusters = [[G.vs[i]["name"] for i in comm] for comm in partition]
    degrees = dict(zip(G.vs["name"], G.degree()))

    return G, partition, clusters, degrees

In [ ]:
def plot_raw_filt_clusters(G, partition, ax):
    layout = G.layout("fr")

    gene_names = None
    if G.vcount() <= 100:
        gene_names = G.vs["name"]

    labels = partition.membership
    n_clusters = len(set(labels))

    palette = ig.drawing.colors.ClusterColoringPalette(n_clusters)
    colors = [palette[label] for label in labels]

    ig.plot(
        G,
        target=ax,
        layout=layout,
        vertex_color=colors,
        vertex_label=gene_names,
        vertex_label_size=6,
        vertex_size=8,
        edge_width=[3 * w for w in G.es["weight"]],
        # edge_color="lightgrey",
        # bbox=(400, 400),
    )
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1)
        spine.set_color("lightgrey")


def graph_metadata(G, sig_corr_df, tissue, ct, clusters, res):
    V = G.vcount()
    E = G.ecount()

    density = 0
    if V > 1:
        density = E / (V * (V - 1) / 2)

    return {
        "tissue.cell": f"{tissue}.{ct}",
        "Vertices": V,
        "Edges": E,
        "density": round(density, 3),
        "res": res,
        "n_clusters": len(clusters),
        "max_dif": round(sig_corr_df.dif.max(), 3),
        "min_dif": round(sig_corr_df.dif.min(), 3),
        "mean_dif": round(sig_corr_df.dif.mean(), 3),
        "median_dif": round(np.median(sig_corr_df.dif), 3),
    }

In [ ]:
def remove_lonely_genes(cluster, degrees, min_cluster_size):
    if len(cluster) <= min_cluster_size:
        return cluster
    return [x for x in cluster if degrees[x] >= round(np.log(len(cluster)))]

| cluster size `n` | `round(ln(n))` |
| ------------------- | -------------- |
| 5 – 12              | 2              |
| 13 – 33             | 3              |
| 34 – 90             | 4              |
| 91 – 244            | 5              |
| 245 – 665           | 6              |
| 666 – 1808          | 7              |
| 1809 – 4000         | 8              |


### Step 3: gene networks & hubs detection

In [5]:
res_dict_path = "./data/senepy_denovo_signatures_code/tc_resolution_dict.pkl"

with open(res_dict_path, "rb") as file:
    gamma_dict = pickle.load(file)

In [ ]:
hubs = {}  # individual signature clusters
signatures = {}  # all genes for tissue_cell that pass filtering
perm_dir = "./data/senepy_denovo_signatures_code/sp_pearson_perm_output"
files = os.listdir(perm_dir)
files.sort()
min_cluster_size = 5
seed = 69

for f in tqdm(files, total=len(files)):
    tissue, ct, _ = f.split(".", maxsplit=2)
    tqdm.write(f"\nProcessing {tissue}.{ct}")
    f_path = os.path.join(perm_dir, f)
    corr_df = pd.read_csv(f_path)
    corr_df["dif"] = corr_df.r - corr_df.rnd_r_q99
    sig_corr_df = corr_df[(corr_df.r > 0) & (corr_df.dif > 0.05)].sort_values(
        "dif", ascending=False
    )

    meta_rows = []

    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    res = gamma_dict[f"{tissue}.{ct}"]

    G, partition, clusters, degrees = build_graph_and_clustering(
        sig_corr_df, resolution=res, seed=seed
    )

    plot_raw_filt_clusters(G, partition, axes[0])
    meta_rows.append(graph_metadata(G, sig_corr_df, tissue, ct, clusters, res))

    clusters = [
        remove_lonely_genes(c, degrees, min_cluster_size) for c in clusters
    ]  # remove genes that are loosely connected to clusters
    clusters = [
        c for c in clusters if len(c) > min_cluster_size
    ]  # remove small clusters

    # combine cluster genes into one signature (set compreh.)
    signature = {g for c in clusters for g in c}

    # filter corrs based on signature
    filtered = sig_corr_df[
        (sig_corr_df.gene1.isin(signature)) & (sig_corr_df.gene2.isin(signature))
    ]

    if len(filtered) == 0:
        tqdm.write("\t❗Граф не прошел фильтрацию")
        meta_df = pd.DataFrame(meta_rows)
        tqdm.write(meta_df.to_string())
        continue
    G_f, partition_f, clusters_f, degrees_f = build_graph_and_clustering(
        filtered, resolution=res, seed=seed
    )
    plot_raw_filt_clusters(G_f, partition_f, axes[1])
    meta_rows.append(graph_metadata(G_f, filtered, tissue, ct, clusters_f, res))

    # избежать пустых/одноэлементных кластеров (не факт что тут такое возмножно, но пусть будет)
    clusters_f = [c for c in clusters_f if len(c) > 1]

    # приводим к форме [(gene, degree), ...]
    # 1) сортируем гены внутри каждого кластера по degree (убывание)
    clusters_w_deg = [
        sorted(
            [(g, degrees_f[g]) for g in c], key=lambda x: x[1], reverse=True
        )  # x = (gene, degree)
        for c in clusters_f
    ]
    # 2) сортируем сами кластеры по размеру (убывание)
    clusters_sorted = sorted(clusters_w_deg, key=lambda c: len(c), reverse=True)

    for i, cluster in enumerate(clusters_sorted):
        hubs[(tissue, ct, i)] = cluster

    # сет, сортированный по key=степень вершины в графе, по убыванию
    signature_f = sorted(
        {g for c in clusters_f for g in c}, key=lambda g: degrees_f[g], reverse=True
    )
    signatures[(tissue, ct)] = [(g, degrees_f[g]) for g in signature_f]

    meta_df = pd.DataFrame(meta_rows)
    tqdm.write(meta_df.to_string())
    display(fig)
    plt.close(fig)  # <--- это предотвращает повторную отрисовку

In [ ]:
# удаление пустых сигнатур и хабов (если нужно)
signatures = {k: v for (k, v) in signatures.items() if len(signatures[k]) != 0}
hubs = {k: v for (k, v) in hubs.items() if len(hubs[k]) != 0}

In [10]:
out_dir = "./data/senepy_denovo_signatures_code/ready"
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [11]:
with open(os.path.join(out_dir, "human_hubs.pkl"), "wb") as file:
    pickle.dump(hubs, file)

with open(os.path.join(out_dir, "human_signatures.pkl"), "wb") as file:
    pickle.dump(signatures, file)